In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("napistu_torch")

In [ ]:
from napistu.gcs import downloads
from napistu.gcs.constants import GCS_ASSETS_NAMES, GCS_SUBASSET_NAMES
path = downloads.load_public_napistu_asset(
    asset=GCS_ASSETS_NAMES.TEST_PATHWAY,
    data_dir="/tmp/napistu_data",
    subasset=GCS_SUBASSET_NAMES.SBML_DFS
)
print(path)

In [ ]:
from napistu_torch.ml.wandb import get_wandb_metrics_table

In [ ]:
wandb_metrics = get_wandb_metrics_table("napistu/napistu-experiments/2d97r0kd")

In [ ]:
from napistu_torch.ml.constants import METRIC_SUMMARIES, DEFAULT_MODEL_CARD_METRICS, METRIC_DISPLAY_NAMES
from napistu_torch.ml.wandb import _get_wandb_run_object

In [ ]:
summary

In [ ]:
run_path = "napistu/napistu-experiments/2d97r0kd"
metrics = None

if metrics is None:
    metrics = DEFAULT_MODEL_CARD_METRICS

# Get WandB run object
run = _get_wandb_run_object(
    run_path=run_path,
)

# Extract metrics from summary
summary = run.summary._json_dict

# Build DataFrame
rows = []
for metric_key in metrics:
    value = summary.get(metric_key)
    display_name = METRIC_DISPLAY_NAMES.get(metric_key, metric_key)
    rows.append(
        {METRIC_VALUE_TABLE.METRIC: display_name, METRIC_VALUE_TABLE.VALUE: value}
    )

df = pd.DataFrame(rows)
if filter_missing_metrics:
    df = df[
        df[METRIC_VALUE_TABLE.VALUE].notna() & (df[METRIC_VALUE_TABLE.VALUE] != 0)
    ]

During my previous post, I discussed self-supervised edge prediction as a way of embedding the genes in a gene-regulatory network.

This allows genes, metabolites, drugs and other vertices to be connected based on their shared connections however, to date, I've only discussed predicting edges using a dot-product head where a pair's support is direct readout of their similarity in embedding space (\textbf{a} x \textbf{b}). This is a superprisingly powerful head but it is still somewhat limiting especially if there are heterogenous types of vertices or qualitatitvely different ways that they interact.

Here, I'll explore more general approaches for learning mappings between A -> B by evaluating more expressive edge prediction heads (like an MLP) and also applying "relation-aware" heads which can learn distinct mappings for different types of edges.

Edge prediction is a powerful approach for predicting regulatory relationships between molecular species but not all regulatory relations are equivalent. They vary both in how a pair of molecules interact (physical, functional, mechanistic) and in terms of what the consequences of this interaction are (activation, inhibition, ambiguous, inert). This information is partially captured by the edge encoder to capture how likely a message is to be transmitted along edges in the graph during message passing but the ultimate prediction is just whether an edge/interaction exists (yes/no?). Ideally, I'd like these models to be able to predict how an interaction occurs and so I turned to relation-aware approaches. Relations are commonly discussed in the context of graph knowledgebases, where we may have a set of qualitatively different vertex types which can be connected by different types of relationships. For example, we could embed the Open Targets knowledgebase graph to organize genes and phenotypes in a common manifold and could also connect other entity types like drugs, chemical probes, etc. Learning relation-aware edges lets us define connections to map between regions of the embedding.

## Transfer learning

To make it easier to evaluate different heads using a common encoding framework I trained a 128-dim GraphConv message passing encoder using a 32-dim edge encoder to learn edge reliability and a simple dot product head.

- deploying to HF; loading from HF
- oncycle to ramp up LR

## Restructuring the NapistuData

- Removing reaction vertices and allowing for multiple directed edges between each pair of molecules
- Adding relation_type. from- and to-sbo term.


## Fitting Relation aware models

- How do inhibitor and stimulator compare to modifier? Are (inhibitor - modifier) and (stimulator - modifier) antiparallel vectors?

## Evaluating regulatory predictions using PerturbSeq

To assess whether signed regulatory predictions (A->B) are meaningful I wanted to compare them to synthetic biology experiments where gene A is perturbed and its impact on gene B can be monitored. For human biology, this is accomplished using PerturbSeq, and its variants like CROP-seq. These experiments generally involve a CRISPR knockdown of a regulator and use single-cell RNAseq to read-out perturbations (and to the decode the perturbation in pooled screens). (As an aside, I worked on a similar problem when developing the yeast [Induction Dynamics Expression Atlast](https://pubmed.ncbi.nlm.nih.gov/32181581/) where we engineered estradiol-inducible promoters to dynamically perturb all of yeast's non-essential transcription factors and then read-out their effects with transcriptome-wide time-series.)

To use PerturbSeq data for evaluation I wanted signed effect predictions which is exactly what Harmonizome produces. Harmonizome constructs gene x whatever (gene, disease, tissue, etc) profiles from a vast array of sources for assessing the similarity of biological conditions.



In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from napistu_torch.configs import ExperimentConfig, ModelConfig, DataConfig, TaskConfig, TrainingConfig
from napistu_torch.load.constants import DEFAULT_ARTIFACTS_NAMES
from napistu_torch.load.napistu_graphs import construct_unlabeled_napistu_data
from napistu_torch.napistu_data_store import NapistuDataStore
from napistu_torch.models.constants import ENCODERS, HEADS
from napistu_torch.lightning.workflows import prepare_experiment, fit_model, resume_experiment



In [ ]:
RELATION_STORE = "./relation_store"
SBML_DFS_PATH = os.path.expanduser("~/Desktop/GITHUB/napistu/dev/napistu_data/human_consensus/human_consensus_no_rxns/sbml_dfs.pkl")
NAPISTU_GRAPH_PATH = os.path.expanduser("~/Desktop/GITHUB/napistu/dev/napistu_data/human_consensus/human_consensus_no_rxns/napistu_graph.pkl")


In [ ]:

#napistu_data_store = NapistuDataStore.create(
#    store_dir = RELATION_STORE,
#    napistu_graph_path = ,
#    sbml_dfs_path = os.path.expanduser("~/Desktop/GITHUB/napistu/dev/napistu_data/human_consensus/human_consensus_no_rxns/sbml_dfs.pkl"),
#    overwrite = True
#)

napistu_data_store = NapistuDataStore(RELATION_STORE)
napistu_data_store.ensure_artifacts([DEFAULT_ARTIFACTS_NAMES.RELATION_PREDICTION, DEFAULT_ARTIFACTS_NAMES.EDGE_STRATA_BY_NODE_TYPE, DEFAULT_ARTIFACTS_NAMES.SPECIES_IDENTIFIERS])

# load species identifiers
species_identifiers = napistu_data_store.load_pandas_df(DEFAULT_ARTIFACTS_NAMES.SPECIES_IDENTIFIERS)

In [ ]:
from napistu_torch.labels.apply import decode_labels

napistu_data = napistu_data_store.load_napistu_data("relation_prediction")

# get a summary of all of the relation types
decoded_labels = decode_labels(napistu_data.relation_type, napistu_data.relation_manager)
relation_type_counts = pd.Series(decoded_labels, name = "relation_type").value_counts().to_frame()
relation_type_counts["total weight [sqrt(count)]"] = (relation_type_counts["count"] / np.sqrt(relation_type_counts["count"])).round(1)
relation_type_counts

In [1]:
import os
from pathlib import Path

import torch
from napistu_torch.evaluation.manager import LocalEvaluationManager, RemoteEvaluationManager
from napistu_torch.load.checkpoints import Checkpoint
from napistu_torch.lightning.full_graph_datamodule import FullGraphDataModule
from napistu_torch.ml.hugging_face import HFModelLoader, HFModelPublisher, generate_model_card

EXPERIMENTS_DIR = Path("~/Desktop/Experiments").expanduser()
EXPERIMENT_SET = "20251220_relation_prediction"
EXPERIMENT_NAME = "relation_prediction_distmult_128e_transfer_v2"

model_path = EXPERIMENTS_DIR / EXPERIMENT_SET / EXPERIMENT_NAME
# model_path = "/Users/sean/Desktop/Experiments/relation_prediction/relation_prediction_rotate_transfer"

evaluation_manager = LocalEvaluationManager(model_path)
# local checkpoint
checkpoint = Checkpoint.load(evaluation_manager.best_checkpoint_path)


/Users/sean/Desktop/GITHUB/napistu/lib/napistu-scrapyard/scraps/pytorch/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# remote evaluation manager
evaluation_manager = RemoteEvaluationManager.from_huggingface(
    repo_id="seanhacks/relation_prediction_distmult_128e",
    data_store_dir = EXPERIMENTS_DIR / ".store"
)

evaluation_manager = RemoteEvaluationManager.from_huggingface(
    repo_id="seanhacks/relation_prediction_distmult_128e",
    data_store_dir = EXPERIMENTS_DIR / ".test_store"
)

In [ ]:
import os
from pathlib import Path

from napistu_torch.configs import ExperimentConfig
from napistu_torch.lightning.workflows import prepare_experiment

# Load your transfer learning config
config = ExperimentConfig.from_yaml(Path(os.path.expanduser("~/Desktop/EXPERIMENTS/relation_prediction/relation_prediction_mlp_256e_transfer_v1.yaml")))

# Prepare the experiment (this loads pretrained weights)
experiment_dict = prepare_experiment(config)
model = experiment_dict['model']


## Evaluation

Predicting changes in Peturb-Seq experiments
- Replogle et al., 2023 (genome-scale PerturbSeq)
- Misc other experiments

### Directly load the Replogle supp

Summaries of the Replogle ddata are quite limited. Differential expression summmaries for one of the K562 datasets are avaialble but fold-changes are NOT signed since diffex was assessed by comparing an observed distribution of guide effects to null guides using an Anderson-Darling test. Still, this is useful for the actual diffex summares and to validate other resources which have re-analyzed the dataset and reported signed-differential exprssion.


In [ ]:
import os

import pandas as pd

from napistu.constants import BQB, IDENTIFIERS, ONTOLOGIES, SBML_DFS
from napistu.matching.species import features_to_pathway_species
from napistu.ingestion.perturbseq import (
    ingest_replogle_pvalues,
    load_harmonizome_perturbseq_datasets,
    load_replogle_pvalues_with_species_ids
)

def _get_distinct_replogle_pvalues(replogle_pvalues_with_species_ids: pd.DataFrame) -> pd.DataFrame:
    """ Reduce the Replogle reported significance to a single entry per perturbed-target pair. """
    
    grouped = replogle_pvalues_with_species_ids.groupby(["perturbed_species_id", "target_species_id"])
    distinct_replogle_pvalues = (
        grouped
        .agg({"pvalue": "min"})
        .assign(n=grouped.size())
        .reset_index()
    )
    return distinct_replogle_pvalues

def _get_distinct_harmonizome_perturbseq_interactions(aggregated_perturbseq_data_with_species_ids: pd.DataFrame) -> pd.DataFrame:
    """ Reduce the harmonizome perturbseq data to a single entry per study-type-perturbed-target pair. """

    grouped = (
        aggregated_perturbseq_data_with_species_ids
        .assign(**{"absolute std value": lambda df: df["Standardized Value"].abs()})
        .sort_values("absolute std value", ascending=False)
        .groupby(["dataset_shortname", "perturbation_study", "perturbation_type", "perturbed_species_id", "target_species_id"])
    )

    distinct_harmonizome_perturbseq_interactions = (
        grouped
        .agg({
            "Standardized Value": "first",
            "Threshold Value": "first",
        })
        .assign(n_interactions=grouped.size())
        .reset_index()
    )

    return distinct_harmonizome_perturbseq_interactions

LOCAL_REPLOGLE_PERTURBSEQ_PATH = "/tmp/repogle_perturbseq.csv.gz"

if not os.path.isfile(LOCAL_REPLOGLE_PERTURBSEQ_PATH):
    # download the supplemental table with K562 pvalues
    ingest_replogle_pvalues(LOCAL_REPLOGLE_PERTURBSEQ_PATH)

# load the supplement and match target and perturbed gene identifiers to species ids
replogle_pvalues_with_species_ids = load_replogle_pvalues_with_species_ids(LOCAL_REPLOGLE_PERTURBSEQ_PATH, species_identifiers)


# define on-target KD as a perturbation where the target and perturbed gene are the same. This is probably conservative relative to the 30% decreased expression criteria used in the Replogle paper
ontarget_kd = replogle_pvalues_with_species_ids.query("target_ensembl_gene == perturbed_ensembl_gene").query("pvalue < 0.1")

# strong perturbations are those with >50 interactions with p < 0.05 (note that these are already BH corrected)
perturbation_effects = replogle_pvalues_with_species_ids.query("perturbed_gene_id in @ontarget_kd.perturbed_gene_id").query("pvalue < 0.05")

perturbation_effect_counts = perturbation_effects.value_counts("perturbed_gene_id")
# this is in the right ballpark of the 1973 strong perturbations reported in the Replogle paper (which also require an on-target effect and a minimal cells impacted count)
strong_perturbation_gene_ids = perturbation_effect_counts[perturbation_effect_counts >= 50].index.tolist()

distinct_replogle_pvalues = _get_distinct_replogle_pvalues(perturbation_effects.query("perturbed_gene_id in @strong_perturbation_gene_ids"))

### Load from Harmonizome


In [ ]:
LOCAL_HARMONIZOME_DATA_DIR = "/tmp/harmonizome_data"

aggregated_perturbseq_data_with_species_ids = load_harmonizome_perturbseq_datasets(LOCAL_HARMONIZOME_DATA_DIR, species_identifiers)
distinct_harmonizome_perturbseq_interactions = _get_distinct_harmonizome_perturbseq_interactions(aggregated_perturbseq_data_with_species_ids)


In [ ]:
# compare reported K562 statistical significance from the Replogle et al. paper to the harmonizome diffex calls

merged_replogle_results = (
    distinct_harmonizome_perturbseq_interactions
    .query("dataset_shortname == 'reploglek562genomewide'")
    .merge(distinct_replogle_pvalues, on = ["perturbed_species_id", "target_species_id"], how = "left")
)

# Create binary indicator for whether p-value exists
merged_replogle_results['has_pvalue'] = merged_replogle_results['pvalue'].notna()

# Create the direction category based on standardized value and threshold
def classify_direction(row):
    std_val = row['Standardized Value']
    threshold = row['Threshold Value']
    
    if abs(std_val) < threshold:
        return 'unchanged'
    elif std_val > 0:
        return 'up'
    else:
        return 'down'

merged_replogle_results['direction'] = merged_replogle_results.apply(classify_direction, axis=1)

# Create the contingency table
contingency_table = pd.crosstab(
    merged_replogle_results['has_pvalue'],
    merged_replogle_results['direction'],
)

print(contingency_table)


In [ ]:
distinct_replogle_pvalues.merge(distinct_harmonizome_perturbseq_interactions, on = ["perturbed_species_id", "target_species_id"], how = "inner")

## EXTRAS

In [ ]:
sbml_dfs = napistu_data_store.load_sbml_dfs()
napistu_graph = napistu_data_store.load_napistu_graph()
napistu_data = construct_unlabeled_napistu_data(sbml_dfs, napistu_graph, splitting_strategy = "edge_mask")
napistu_graph.show_summary()

x = sbml_dfs.get_characteristic_species_ids(dogmatic = False)

In [ ]:
x

In [ ]:
# define the relations
# from > to sbo_term_names

In [ ]:
# setting up the experiment
from napistu_torch.load.gcs import gcs_model_to_store
from napistu_torch.configs import ExperimentConfig, ModelConfig, DataConfig, TaskConfig, TrainingConfig
from napistu_torch.models.constants import ENCODERS, HEADS

In [ ]:
napistu_data = napistu_data_store.load_napistu_data("edge_prediction")
edge_strata = napistu_data_store.load_pandas_df("edge_strata_by_node_type")
extended_edge_strata = napistu_data_store.load_pandas_df("edge_strata_by_node_species_type")